# Data collection: Argentina vs Egypt, World Cup 2026

Goal: collect the public FIFA data needed for a first open momentum proxy: match metadata, live payload, event timeline, and official hydration break intervals.

This notebook intentionally starts simple and uses only Python's standard library. Later notebooks can introduce pandas/plotting once the data shape is stable.

In [ ]:
import json
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://api.fifa.com/api/v3"
LANGUAGE = "en"
ID_COMPETITION = "17"
ID_SEASON = "285023"

def fetch_json(url: str, output_path: Path) -> dict:
    request = Request(url, headers={"User-Agent": "world-cup-momentum-breaks/0.1"})
    with urlopen(request) as response:
        payload = json.loads(response.read().decode("utf-8"))
    output_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    return payload

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def localized_description(items, default=""):
    if not items:
        return default
    return items[0].get("Description", default)

## 1. Collect the WC26 calendar

The calendar endpoint gives us all World Cup 2026 matches, including the FIFA identifiers needed for the live and timeline endpoints.

In [ ]:
calendar_url = (
    f"{BASE_URL}/calendar/matches?language={LANGUAGE}"
    f"&idCompetition={ID_COMPETITION}&idSeason={ID_SEASON}&count=200"
)
calendar_path = RAW_DIR / "wc26_calendar_matches.json"

if calendar_path.exists():
    calendar = load_json(calendar_path)
else:
    calendar = fetch_json(calendar_url, calendar_path)

len(calendar["Results"]), calendar_path

## 2. Find Argentina vs Egypt

We filter by team abbreviations so the notebook documents how the target match was selected.

In [ ]:
def team_summary(match, side):
    team = match.get(side)
    if not team:
        return None
    return {
        "side": side,
        "id_team": team.get("IdTeam"),
        "abbr": team.get("Abbreviation"),
        "name": team.get("ShortClubName"),
        "score": team.get("Score"),
    }

candidate_matches = []
for match in calendar["Results"]:
    teams = [team_summary(match, "Home"), team_summary(match, "Away")]
    abbreviations = {team["abbr"] for team in teams if team}
    if {"ARG", "EGY"}.issubset(abbreviations):
        candidate_matches.append({
            "match_number": match.get("MatchNumber"),
            "id_match": match.get("IdMatch"),
            "id_stage": match.get("IdStage"),
            "date": match.get("Date"),
            "stage": localized_description(match.get("StageName")),
            "home": team_summary(match, "Home"),
            "away": team_summary(match, "Away"),
        })

candidate_matches

The selected match is Argentina vs Egypt, Match 95, Round of 16.

In [ ]:
MATCH = candidate_matches[0]
ID_MATCH = MATCH["id_match"]
ID_STAGE = MATCH["id_stage"]

MATCH

## 3. Collect match-level FIFA payloads

In [ ]:
live_url = f"{BASE_URL}/live/football/{ID_COMPETITION}/{ID_SEASON}/{ID_STAGE}/{ID_MATCH}?language={LANGUAGE}"
timeline_url = f"{BASE_URL}/timelines/{ID_MATCH}?language={LANGUAGE}"
match_calendar_url = f"{BASE_URL}/calendar/{ID_MATCH}?language={LANGUAGE}"

live_path = RAW_DIR / "argentina_egypt_400021528_live.json"
timeline_path = RAW_DIR / "argentina_egypt_400021528_timeline.json"
match_calendar_path = RAW_DIR / "argentina_egypt_400021528_calendar.json"

live = load_json(live_path) if live_path.exists() else fetch_json(live_url, live_path)
timeline = load_json(timeline_path) if timeline_path.exists() else fetch_json(timeline_url, timeline_path)
match_calendar = load_json(match_calendar_path) if match_calendar_path.exists() else fetch_json(match_calendar_url, match_calendar_path)

{
    "live_path": str(live_path),
    "timeline_path": str(timeline_path),
    "match_calendar_path": str(match_calendar_path),
    "timeline_events": len(timeline.get("Event", [])),
}

## 4. Quick data audit

In [ ]:
match_summary = {
    "match": f"{live['HomeTeam']['ShortClubName']} {live['HomeTeam']['Score']}-{live['AwayTeam']['Score']} {live['AwayTeam']['ShortClubName']}",
    "stage": localized_description(live.get("StageName")),
    "date_utc": live.get("Date"),
    "stadium": localized_description(live.get("Stadium", {}).get("Name")),
    "match_time": live.get("MatchTime"),
    "timeline_events": len(timeline.get("Event", [])),
}

match_summary

In [ ]:
event_type_counts = Counter(
    localized_description(event.get("TypeLocalized"), "Unknown")
    for event in timeline.get("Event", [])
)

event_type_counts.most_common()

## 5. Extract hydration break intervals

FIFA records hydration breaks as a `Delay` event followed by a `Resume` event. These intervals will be shaded on the momentum chart in a later notebook.

In [ ]:
def event_text(event):
    return " ".join(description.get("Description", "") for description in event.get("EventDescription", []))

hydration_intervals = []
open_break = None

for event in timeline.get("Event", []):
    label = localized_description(event.get("TypeLocalized"))
    text = event_text(event)
    lower_text = f"{label} {text}".lower()

    if "hydration break" in lower_text and label == "Delay":
        open_break = event
    elif open_break and label == "Resume":
        hydration_intervals.append({
            "start_minute": open_break.get("MatchMinute"),
            "start_timestamp": open_break.get("Timestamp"),
            "end_minute": event.get("MatchMinute"),
            "end_timestamp": event.get("Timestamp"),
            "period": open_break.get("Period"),
            "start_event_id": open_break.get("EventId"),
            "end_event_id": event.get("EventId"),
        })
        open_break = None

hydration_intervals

## Notes for the next step

The next notebook should turn the timeline into an event table with normalized match minutes and event weights. That gives us the first open momentum proxy.